# 20260828 新规范全量验收审计
## tl;dr
沪深完整运行共 26,224,341 项匹配、23 项停牌阶段排除；不匹配、数据异常和来源不足均为零。原始状态分布与 Rust 审计计数一致。四个代码单元已按顺序执行，输出保存在本文件。
## Context & Methods
读取本轮 Rust 验证器完整扫描输出，不合并历史报告。粒度为市场×证券×参考帧（静态阶段每证券一项）。
### Key Assumptions
输入为 `/hdd/data/stock/raw_level2_parquet/date=20260828`；SH MarketData、SZ mdl_6_28_0 作独立参考。默认标准窗口，不传诊断覆盖参数。停牌排除不是匹配，数据错误和来源不足阻断验收。规范见 `docs/snapshot-validation-rules.md`。
只审计 JSON 汇总及异常证据；并非第二套订单簿计算器。真实计算由 qtp-replay 完成。

## Data
读取两份本轮完整报告；源码状态复核使用 Clara 的 pyarrow 环境。复现完整命令及二进制 SHA-256 见验证记录。

In [1]:
from pathlib import Path
from collections import Counter
import json
root = Path.cwd() if (Path.cwd() / 'Cargo.toml').exists() else Path.cwd().parent
reports = {m: json.loads((root / f'reports/20260828-{m.lower()}-standard-aligned-full.json').read_text()) for m in ('SH', 'SZ')}
for market, report in reports.items():
    assert not report['diagnostic_window_override']
    assert report['total_anchors'] == report['matched'] + report['mismatched'] + report['excluded_by_status'] + report['data_errors'] + report['missing_source']
    assert report['total_anchors'] == sum(c['total'] for c in report['breakdown'].values())
    assert report['matched'] == sum(c['matched'] for c in report['breakdown'].values())
    print(market, {key: report[key] for key in ('total_anchors', 'matched', 'mismatched', 'excluded_by_status', 'data_errors', 'missing_source', 'match_rate')})


SH {'total_anchors': 12342476, 'matched': 12342471, 'mismatched': 0, 'excluded_by_status': 5, 'data_errors': 0, 'missing_source': 0, 'match_rate': 1.0}
SZ {'total_anchors': 13881888, 'matched': 13881870, 'mismatched': 0, 'excluded_by_status': 18, 'data_errors': 0, 'missing_source': 0, 'match_rate': 1.0}


## Results
分别报告分母、匹配、差异、状态排除及数据质量问题，不能仅看可比匹配率。

In [2]:
for market, report in reports.items():
    print('\n', market, 'effective windows', dict(Counter(report['continuous_lookahead_ms_by_symbol'].values())))
    for segment, counts in sorted(report['breakdown'].items()):
        print(segment, counts)
    print('Rule tags:', report['match_tags'])
    print('Post-close events:', report['replay']['sz_after_close_events'])
    print('Status counts:', dict(sum((Counter(audit['status_counts']) for audit in report['phase_audit']), Counter())))



 SH effective windows {1000: 3095}
etf.continuous_trading {'total': 2793370, 'comparable': 2793370, 'matched': 2793370, 'mismatched': 0, 'not_comparable': 0, 'excluded_by_status': 0, 'data_errors': 0, 'missing_source': 0}
etf.market_close {'total': 780, 'comparable': 780, 'matched': 780, 'mismatched': 0, 'not_comparable': 0, 'excluded_by_status': 0, 'data_errors': 0, 'missing_source': 0}
etf.pre_open {'total': 780, 'comparable': 775, 'matched': 775, 'mismatched': 0, 'not_comparable': 5, 'excluded_by_status': 5, 'data_errors': 0, 'missing_source': 0}
stock.continuous_trading {'total': 9542916, 'comparable': 9542916, 'matched': 9542916, 'mismatched': 0, 'not_comparable': 0, 'excluded_by_status': 0, 'data_errors': 0, 'missing_source': 0}
stock.market_close {'total': 2315, 'comparable': 2315, 'matched': 2315, 'mismatched': 0, 'not_comparable': 0, 'excluded_by_status': 0, 'data_errors': 0, 'missing_source': 0}
stock.pre_open {'total': 2315, 'comparable': 2315, 'matched': 2315, 'mismatched'

In [3]:
for market, report in reports.items():
    print('\n', market, 'field differences', report['mismatch_fields'])
    for outcome in ('excluded_by_status', 'data_error', 'missing_source', 'mismatched'):
        rows = [r for r in report['records'] if r['outcome'].replace('_', '').lower() == outcome.replace('_', '').lower()]
        print(outcome, 'detail rows:', len(rows), 'symbols:', sorted({r['symbol'] for r in rows})[:50])
        for row in rows[:5]:
            print({key: row.get(key) for key in ('symbol', 'anchor', 'reason', 'reference_time_ms', 'best_candidate_time_ms', 'best_candidate_raw_sequence', 'differences')})
    assert report['omitted_mismatched_records'] == 0
    assert report['omitted_not_comparable_records'] == 0



 SH field differences {}
excluded_by_status detail rows: 5 symbols: ['513100', '513300', '513310', '513500', '513870']
{'symbol': '513100', 'anchor': 'pre_open', 'reason': 'excluded phase status SUSP', 'reference_time_ms': None, 'best_candidate_time_ms': None, 'best_candidate_raw_sequence': None, 'differences': []}
{'symbol': '513300', 'anchor': 'pre_open', 'reason': 'excluded phase status SUSP', 'reference_time_ms': None, 'best_candidate_time_ms': None, 'best_candidate_raw_sequence': None, 'differences': []}
{'symbol': '513310', 'anchor': 'pre_open', 'reason': 'excluded phase status SUSP', 'reference_time_ms': None, 'best_candidate_time_ms': None, 'best_candidate_raw_sequence': None, 'differences': []}
{'symbol': '513500', 'anchor': 'pre_open', 'reason': 'excluded phase status SUSP', 'reference_time_ms': None, 'best_candidate_time_ms': None, 'best_candidate_raw_sequence': None, 'differences': []}
{'symbol': '513870', 'anchor': 'pre_open', 'reason': 'excluded phase status SUSP', 'refe

### 独立状态分布复核
需要 pyarrow；仅投影状态、时间、证券代码和来源行号。与 Rust 审计计数核对，不使用盘口数据重建订单簿。

In [4]:
import sys
sys.path.insert(0, str(root / 'analysis'))
from audit_snapshot_phases import profile
for market, report in reports.items():
    source = profile(market)
    independent = sum((Counter(c) for c in source['status_counts'].values()), Counter())
    rust = sum((Counter(a['status_counts']) for a in report['phase_audit']), Counter())
    assert independent == rust
    print(market, 'source rows', source['source_rows'], 'selected symbols', source['selected_symbols'])
    print('Order errors:', source['order_errors'])
    print('Excluded statuses:', source['excluded_status_symbols'])
    print('State transitions:', source['excluded_symbol_transitions'])


SH source rows 14272071 selected symbols 3095
Order errors: Counter()
Excluded statuses: {'SUSP': ['513100', '513300', '513310', '513500', '513870']}
State transitions: {'513100': [('08:45:27.000', 'SUSP'), ('10:30:00.000', 'TRADE'), ('14:57:00.000', 'CCALL'), ('15:00:03.000', 'CLOSE'), ('15:05:03.000', 'ENDTR')], '513300': [('08:45:25.000', 'SUSP'), ('10:30:01.000', 'TRADE'), ('14:57:01.000', 'CCALL'), ('15:00:04.000', 'CLOSE'), ('15:05:01.000', 'ENDTR')], '513310': [('08:45:28.000', 'SUSP'), ('10:30:01.000', 'TRADE'), ('14:57:01.000', 'CCALL'), ('15:00:01.000', 'CLOSE'), ('15:05:01.000', 'ENDTR')], '513500': [('08:45:08.000', 'SUSP'), ('10:30:02.000', 'TRADE'), ('14:57:02.000', 'CCALL'), ('15:00:02.000', 'CLOSE'), ('15:05:02.000', 'ENDTR')], '513870': [('08:45:29.000', 'SUSP'), ('10:30:02.000', 'TRADE'), ('14:57:02.000', 'CCALL'), ('15:00:02.000', 'CLOSE'), ('15:05:02.000', 'ENDTR')]}
SZ source rows 15722868 selected symbols 3612
Order errors: Counter()
Excluded statuses: {'B1': ['00

## Takeaways
本轮全部可比项通过，没有待分析的不匹配。23 个排除项来自 5 只沪市停牌 ETF、4 只深市全天停牌股票和 6 只深市临时停牌 ETF 的对应阶段；复牌后的正常阶段全部通过。详情见 `docs/real-data-validation.md`。单日汇总盘口通过不证明跨日期规则，也不替代 FIFO 和逐订单不变量测试。